In [ ]:
from pathlib import Path

ordner = Path.home() / "Desktop" / "Modul 5" / "Aufgaben"

for datei in ordner.rglob("bestellungen_pii.csv"):
    print("Datei gefunden:", datei)

In [ ]:
import pandas as pd

# CSV-Datei laden
df = pd.read_csv(
    "~/Desktop/Modul 5/Aufgaben/Woche_2/bestellungen_pii.csv"
)

# Struktur prüfen
print("Zeilen und Spalten:", df.shape)
print("\nSpaltennamen:")
print(df.columns.tolist())

In [ ]:
# PII-Inventur: Vollständigkeit und Datentypen prüfen
inventur = pd.DataFrame({
    "Datentyp": df.dtypes.astype(str),
    "Vorhandene Werte": df.notna().sum(),
    "Fehlende Werte": df.isna().sum(),
    "Eindeutige Werte": df.nunique(dropna=True)
})

display(inventur)

In [ ]:
# Re-Identifikations-Check: PLZ + Geburtsdatum

# Nur Datensätze mit vorhandenem Geburtsdatum untersuchen
check = df.dropna(subset=["plz", "geburtsdatum"]).copy()

# Anzahl der Bestellungen pro Kombination ermitteln
kombinationen = check.groupby(
    ["plz", "geburtsdatum"]
)["bestell_id"].transform("count")

# Einmalige Kombinationen zählen
eindeutig = (kombinationen == 1).sum()

print("Bestellungen insgesamt:", len(df))
print("Bestellungen mit PLZ und Geburtsdatum:", len(check))
print("Davon einmalige Kombinationen:", eindeutig)
print("Anteil einmaliger Kombinationen:",
      round(eindeutig / len(check) * 100, 2), "%")

In [ ]:
# Re-Identifikations-Check auf Personenebene

personen = df.drop_duplicates(subset=["kunde_email"]).copy()

personen_check = personen.dropna(
    subset=["plz", "geburtsdatum"]
).copy()

# Kombinationen pro Person zählen
haeufigkeit = personen_check.groupby(
    ["plz", "geburtsdatum"]
)["kunde_email"].transform("count")

eindeutige_personen = (haeufigkeit == 1).sum()

print("Personen insgesamt:", len(personen))
print("Personen mit PLZ und Geburtsdatum:", len(personen_check))
print("Davon eindeutig unterscheidbar:", eindeutige_personen)

if len(personen_check) > 0:
    print(
        "Anteil:",
        round(eindeutige_personen / len(personen_check) * 100, 2),
        "%"
    )

In [ ]:
# Prüfen, ob sich PLZ oder Geburtsdatum
# innerhalb derselben E-Mail-Adresse unterscheiden

abweichungen = df.groupby("kunde_email").agg(
    unterschiedliche_plz=("plz", "nunique"),
    unterschiedliche_geburtsdaten=("geburtsdatum", "nunique")
)

auffaellige_personen = (
    (abweichungen["unterschiedliche_plz"] > 1)
    | (abweichungen["unterschiedliche_geburtsdaten"] > 1)
).sum()

print("Unterschiedliche E-Mail-Adressen:", len(abweichungen))
print("E-Mail-Adressen mit abweichenden Angaben:", auffaellige_personen)

In [ ]:
# PII-Register für den Datenabnehmer Marketing

pii_register = pd.DataFrame([
    ["bestell_id", "Indirekter Identifikator", "Pseudonymisieren", "Kann Bestellungen einer Person zuordnen"],
    ["datum", "Quasi-Identifikator", "Behalten", "Für zeitliche Auswertungen erforderlich"],
    ["kunde_name", "Direkter Identifikator", "Entfernen", "Für Marketingauswertungen nicht erforderlich"],
    ["kunde_email", "Direkter Identifikator", "Pseudonymisieren", "Nur für stabile interne Zuordnung verwenden"],
    ["kunde_telefon", "Direkter Identifikator", "Entfernen", "Für Auswertungen nicht erforderlich"],
    ["strasse", "Direkter Identifikator", "Entfernen", "Ermöglicht Rückschlüsse auf Wohnadressen"],
    ["plz", "Quasi-Identifikator", "Entfernen", "Zusammen mit Geburtsdatum re-identifizierend"],
    ["ort", "Quasi-Identifikator", "Prüfen", "Geografische Auswertung eventuell erforderlich"],
    ["geburtsdatum", "Quasi-Identifikator", "Altersgruppen bilden", "Exaktes Geburtsdatum nicht erforderlich"],
    ["ip_adresse", "Online-Identifikator", "Entfernen", "Für Marketingauswertungen nicht erforderlich"],
    ["produkt", "Keine direkte PII", "Behalten", "Für Produktanalysen erforderlich"],
    ["menge", "Keine direkte PII", "Behalten", "Für Mengenanalysen erforderlich"],
    ["preis_eur", "Keine direkte PII", "Behalten", "Für Umsatzanalysen erforderlich"]
], columns=["Spalte", "Einstufung", "Behandlung", "Begründung"])

display(pii_register)

In [ ]:
 # Arbeitskopie für die Datenschutzbehandlung erstellen
df_sicher = df.copy()

# Umsatz vor der Bearbeitung berechnen
umsatz_vorher = (
    df_sicher["menge"] * df_sicher["preis_eur"]
).sum()

print("Bestellungen:", len(df_sicher))
print("Spalten:", len(df_sicher.columns))
print("Umsatz vorher:", round(umsatz_vorher, 2), "€")

In [ ]:
import uuid

# Eindeutige Kunden ermitteln und zufällige IDs vergeben
kunden_mapping = pd.DataFrame({
    "kunde_email": df_sicher["kunde_email"].dropna().unique()
})

kunden_mapping["kunden_pseudo_id"] = [
    str(uuid.uuid4()) for _ in range(len(kunden_mapping))
]

# Pseudonyme den Bestellungen zuordnen
df_sicher = df_sicher.merge(
    kunden_mapping,
    on="kunde_email",
    how="left",
    validate="m:1"
)

# Nur technische Prüfwerte ausgeben
print("Erzeugte Kunden-IDs:", len(kunden_mapping))
print("Eindeutige Pseudonyme:", kunden_mapping["kunden_pseudo_id"].nunique())
print("Bestellungen nach Zuordnung:", len(df_sicher))
print("Bestellungen ohne Pseudonym:", df_sicher["kunden_pseudo_id"].isna().sum())

In [ ]:
# Altersgruppen aus Geburtsdaten erstellen
geburt = pd.to_datetime(
    df_sicher["geburtsdatum"],
    errors="coerce"
)

# Alter zum Stichtag 27.10.2026 berechnen
stichtag = pd.Timestamp("2026-10-27")

alter = (
    stichtag.year
    - geburt.dt.year
    - (
        (geburt.dt.month > stichtag.month)
        | (
            (geburt.dt.month == stichtag.month)
            & (geburt.dt.day > stichtag.day)
        )
    )
)

# Altersgruppen bilden
df_sicher["altersgruppe"] = pd.cut(
    alter,
    bins=[0, 19, 29, 39, 49, 59, 69, 120],
    labels=[
        "Unter 20", "20-29", "30-39", "40-49",
        "50-59", "60-69", "70+"
    ],
    include_lowest=True
)

df_sicher["altersgruppe"] = (
    df_sicher["altersgruppe"]
    .cat.add_categories("Unbekannt")
    .fillna("Unbekannt")
)

# Nur aggregierte Ergebnisse anzeigen
print(df_sicher["altersgruppe"].value_counts())

In [ ]:
# Daten für Marketing zusammenstellen
marketing_df = df_sicher[
    [
        "kunden_pseudo_id",
        "datum",
        "produkt",
        "menge",
        "preis_eur",
        "altersgruppe"
    ]
].copy()

# Struktur kontrollieren, ohne persönliche Daten auszugeben
print("Bestellungen:", len(marketing_df))
print("Spalten:", len(marketing_df.columns))

print("\nVerbleibende Spalten:")
print(marketing_df.columns.tolist())

print("\nUmsatz nach Bereinigung:")
umsatz_nachher = (
    marketing_df["menge"] * marketing_df["preis_eur"]
).sum()

print(round(umsatz_nachher, 2), "€")

In [ ]:
# Datenschutzprüfung: unerwünschte Originalspalten

verbotene_spalten = [
    "bestell_id",
    "kunde_name",
    "kunde_email",
    "kunde_telefon",
    "strasse",
    "plz",
    "ort",
    "geburtsdatum",
    "ip_adresse"
]

# Prüfen, ob unerwünschte Spalten vorhanden sind
gefundene_pii = [
    spalte for spalte in verbotene_spalten
    if spalte in marketing_df.columns
]

assert len(gefundene_pii) == 0, (
    f"Unerwünschte Spalten gefunden: {gefundene_pii}"
)

print("Datenschutzprüfung erfolgreich!")
print("Unerwünschte Originalspalten:", len(gefundene_pii))
print("Bestellungen:", len(marketing_df))
print("Eindeutige Kundenpseudonyme:",
      marketing_df["kunden_pseudo_id"].nunique())

In [ ]:
from pathlib import Path

# Separater Ordner außerhalb des GitHub-Projekts
schutzordner = Path.home() / "Private_Daten" / "Modul_5"

# Ordner erstellen und Zugriff auf den Benutzer beschränken
schutzordner.mkdir(parents=True, exist_ok=True)
schutzordner.chmod(0o700)

print("Geschützter Ordner vorbereitet.")
print("Ordner:", schutzordner)
print("Liegt außerhalb des GitHub-Projekts: Ja")

In [ ]:
# Zuordnungstabelle getrennt vom Marketing-Datensatz speichern
mapping_datei = schutzordner / "kunden_mapping.csv"

kunden_mapping.to_csv(
    mapping_datei,
    index=False
)

# Dateizugriff auf den Benutzer beschränken
mapping_datei.chmod(0o600)

# Speicherung kontrollieren
print("Zuordnungstabelle gespeichert:", mapping_datei.exists())
print("Anzahl Zuordnungen:", len(kunden_mapping))
print("Dateiberechtigung:", oct(mapping_datei.stat().st_mode & 0o777))

In [ ]:
# Gespeicherte Zuordnungstabelle überprüfen

mapping_kontrolle = pd.read_csv(mapping_datei)

assert len(mapping_kontrolle) == 64
assert mapping_kontrolle["kunde_email"].is_unique
assert mapping_kontrolle["kunden_pseudo_id"].is_unique

# Prüfen, ob beim Speichern Werte verändert wurden
assert mapping_kontrolle.equals(kunden_mapping)

print("✅ Zuordnungstabelle erfolgreich eingelesen")
print("✅ 64 eindeutige Kundenzuordnungen")
print("✅ Keine doppelten E-Mail-Adressen")
print("✅ Keine doppelten Pseudonyme")
print("✅ Gespeicherte Daten stimmen mit dem Original überein")

In [ ]:
# Abschlussprüfung – Woche 2, Tag 3

# 1. Alle Bestellungen erhalten
assert len(marketing_df) == 80

# 2. Genau 64 Kundenpseudonyme
assert marketing_df["kunden_pseudo_id"].nunique() == 64

# 3. Keine fehlenden Pseudonyme
assert marketing_df["kunden_pseudo_id"].notna().all()

# 4. Keine unerwünschten Originalspalten
assert not set(verbotene_spalten).intersection(marketing_df.columns)

# 5. Umsatz unverändert
assert abs(umsatz_vorher - umsatz_nachher) < 0.01

# 6. Altersgruppen vollständig
assert marketing_df["altersgruppe"].notna().all()

# 7. Zuordnungstabelle separat gespeichert
assert mapping_datei.exists()

print("✅ Alle Qualitätsprüfungen bestanden!")
print("✅ 80 Bestellungen erhalten")
print("✅ 64 eindeutige Kundenpseudonyme")
print("✅ Keine fehlenden Pseudonyme")
print("✅ Keine unerwünschten Originalspalten")
print("✅ Umsatz unverändert: 12.226,10 €")
print("✅ Altersgruppen vollständig")
print("✅ Zuordnungstabelle separat gespeichert")

## Datenschutz- und Copilot-Reflexion

### 1. Re-Identifikationsrisiko

Der Ausgangsdatensatz enthält 80 Bestellungen und 64 unterschiedliche E-Mail-Adressen.

Bei 23 Kunden mit vorhandener Postleitzahl und Geburtsdatum war jede Kombination innerhalb des untersuchten Datensatzes einzigartig.

Unter der Annahme, dass jede E-Mail-Adresse einer Person entspricht, waren somit 100 % dieser 23 Kunden anhand der beiden Merkmale unterscheidbar.

Das bedeutet nicht, dass ihre Identität automatisch bekannt ist. Zusätzliche Informationen könnten jedoch eine Wiedererkennung ermöglichen.

### 2. Datenminimierung für Marketing

Für die Marketingauswertung wurden nur sechs Spalten übernommen:

- kunden_pseudo_id
- datum
- produkt
- menge
- preis_eur
- altersgruppe

Namen, E-Mail-Adressen, Telefonnummern, Anschriften, Postleitzahlen, Geburtsdaten und IP-Adressen wurden aus dem Marketing-Datensatz entfernt.

Exakte Geburtsdaten wurden durch Altersgruppen ersetzt.

### 3. Pseudonymisierung

Für 64 Kunden wurden eindeutige, zufällig erzeugte UUIDs verwendet.

Mehrere Bestellungen desselben Kunden erhalten dieselbe pseudonyme ID.

Die Zuordnungstabelle wurde getrennt vom Marketing-Datensatz gespeichert und nicht für eine Veröffentlichung vorgesehen.

Die Daten sind pseudonymisiert, aber nicht anonymisiert. Auch Kombinationen aus Bestelldatum, Produkt und Altersgruppe können Rückschlüsse ermöglichen.

### 4. Copilot-Reflexion

Ein KI-Assistent könnte vorschlagen, E-Mail-Adressen lediglich durch Hashwerte zu ersetzen.

Ein einfacher Hash ohne geheimen Schlüssel wäre jedoch nicht automatisch ausreichend, da bekannte oder erratbare E-Mail-Adressen durch Vergleich wiedererkannt werden könnten.

Ebenso wäre es problematisch, nur Namen und Telefonnummern zu entfernen und Postleitzahlen sowie Geburtsdaten unverändert weiterzugeben.

Deshalb wurden zufällige Kundenpseudonyme verwendet, die Zuordnungstabelle getrennt gespeichert und nicht benötigte Originalspalten entfernt.

Diese Beispiele beschreiben mögliche KI-Vorschläge und keine tatsächlich beobachteten Copilot-Antworten.

### 5. Qualitätskontrolle

Alle 80 Bestellungen sind erhalten geblieben.

Die 64 Kundenpseudonyme sind eindeutig, und keine Bestellung hat eine fehlende pseudonyme ID.

Der Gesamtumsatz beträgt vor und nach der Bearbeitung 12.226,10 €.

Alle technischen Qualitätsprüfungen wurden erfolgreich bestanden.

### Fazit

Datenschutz beginnt bereits bei der Auswahl der benötigten Informationen.

Pseudonymisierung reduziert Risiken, ersetzt jedoch weder eine Prüfung der Re-Identifizierbarkeit noch angemessene technische und organisatorische Schutzmaßnahmen.

## Hinweise zur Ausführung und zum Datenschutz

Dieses Notebook demonstriert die Pseudonymisierung personenbezogener Bestelldaten.

### Benötigte Eingabedatei

Für die vollständige Ausführung wird die Datei `bestellungen_pii.csv` benötigt.

Diese Datei enthält personenbezogene Daten und wird deshalb nicht im öffentlichen GitHub-Repository bereitgestellt.

Der im Notebook verwendete Dateipfad muss bei einer Ausführung auf einem anderen Rechner angepasst werden.

### Umgang mit der Zuordnungstabelle

Die Datei `kunden_mapping.csv` enthält die Zuordnung zwischen ursprünglichen E-Mail-Adressen und zufällig erzeugten Kundenpseudonymen.

Sie wird ausschließlich lokal in einem separaten Verzeichnis gespeichert und nicht veröffentlicht.

### Einschränkungen

Der Marketing-Datensatz ist pseudonymisiert, nicht anonymisiert.

Die Ergebnisse dienen der Demonstration von Datenschutzmaßnahmen im Rahmen einer Übungsaufgabe.

Die technische Qualitätsprüfung ersetzt keine vollständige Datenschutz- oder DSGVO-Prüfung.